# Nhóm 6 — Verifier-Guided Repair: chạy thực nghiệm trên Kaggle

**Cấu hình notebook:** Accelerator = *None* (CPU, 4 vCPU, ~30 GB RAM) · **Internet = On** · Persistence = Files only.
**Secrets** (Add-ons → Secrets): `GEMINI_API_KEY` (bật cho notebook này).

Quy tắc:
- Chạy lần lượt từng cell. Mỗi bước ghi run record vào `results/raw/` (provenance đầy đủ).
- **Không chạy cell LLM lớn (mục 5)** trước khi trưởng nhóm duyệt dự toán từ pilot (mục 4).
- Phiên Kaggle tối đa 12 giờ → cuối phiên luôn chạy mục 6 để lưu kết quả vào `/kaggle/working`.

## 0. Lấy code (nhánh thực nghiệm)

In [ ]:
%cd /root
!rm -rf repo && git clone -q --depth 1 -b feat/verifier-guided-pipeline https://github.com/tamdv/ai4math-verifier-guided-repair.git repo
%cd /root/repo
!git log --oneline -1
import os; os.environ["PATH"] = os.path.expanduser("~/.elan/bin") + ":" + os.environ["PATH"]

## 1. Cài môi trường (≈10–15 phút: Lean, Mathlib cache, REPL, test)

In [ ]:
!bash scripts/kaggle_setup.sh

In [ ]:
!python scripts/check_llm.py   # 1 lần gọi rất nhỏ (< 1 đồng) để kiểm tra key

## 2. Dữ liệu: seed → kiểm tra bỏ comment → sinh dataset lỗi (4 worker)

In [ ]:
!python scripts/build_seeds.py --n 3000 --workers 4

In [ ]:
!python scripts/check_strip_comments.py --workers 4

In [ ]:
!python scripts/generate_error_dataset.py --workers 4 --max-per-seed 5
!wc -l data/errors/*.jsonl

## 3. Thí nghiệm miễn phí (chỉ Lean): baseline sàn + rule + scalability

In [ ]:
# Sàn (không sửa) và rule-only trên 1K hold-out
!python scripts/run_experiment.py --data data/errors/errors_1k.jsonl --split holdout --strategy none --budget 0 --workers 4
!python scripts/run_experiment.py --data data/errors/errors_1k.jsonl --split holdout --strategy rule_only --budget 3 --workers 4

In [ ]:
# Scalability ($0): 3 mức dữ liệu (100 -> 1K -> 10K) x 3 mức worker x 3 lần lặp
for data in ["errors_100", "errors_1k", "errors_10k"]:
    for w in [1, 2, 4]:
        for r in [1, 2, 3]:
            !python scripts/run_experiment.py --data data/errors/{data}.jsonl --split all --strategy rule_only --budget 1 --workers {w} --repeat-index {r} --tag scale

## 4. Pilot LLM (≈1.300đ) → dự toán đã hiệu chỉnh
Gửi kết quả 2 cell dưới cho trưởng nhóm duyệt trước khi chạy mục 5.

In [ ]:
!python scripts/run_experiment.py --data data/errors/errors_1k.jsonl --split holdout --limit 50 --strategy blind --budget 1 --workers 4 --run-cap 0.10 --tag pilot
!python scripts/run_experiment.py --data data/errors/errors_1k.jsonl --split holdout --limit 50 --strategy llm_only --budget 1 --workers 4 --run-cap 0.10 --tag pilot

In [ ]:
!python scripts/estimate_cost.py --data data/errors/errors_10k.jsonl

## 5. Thí nghiệm LLM chính (CHỈ chạy sau khi đã duyệt dự toán)

In [ ]:
APPROVED = False   # đổi thành True sau khi trưởng nhóm duyệt
assert APPROVED, "Chưa duyệt dự toán"
for r in [1, 2, 3]:   # lần 2-3 dùng cache câu trả lời => ~0đ
    for strat, b in [("blind", 1), ("llm_only", 1), ("hybrid", 1), ("llm_only", 3), ("hybrid", 3)]:
        !python scripts/run_experiment.py --data data/errors/errors_1k.jsonl --split holdout --strategy {strat} --budget {b} --workers 4 --repeat-index {r} --tag rq
!python scripts/run_experiment.py --data data/errors/errors_10k.jsonl --split holdout --strategy hybrid --budget 1 --workers 4 --run-cap 2.0 --tag scale-llm
!python scripts/run_experiment.py --data data/errors/errors_1k.jsonl --split holdout --strategy llm_only --budget 1 --workers 4 --model gemini-3.6-flash --tag strong-model

## 6. Lưu kết quả (luôn chạy cuối phiên)
Tải `out.zip` từ tab Output và gửi lại để commit (run records, ledger chi phí, dataset nhỏ).

In [ ]:
!mkdir -p /kaggle/working/out && cp -r results data/errors /kaggle/working/out/ 2>/dev/null; cp data_sample/errors_sample_100.jsonl /kaggle/working/out/ 2>/dev/null
!cp data/seeds/*.jsonl /kaggle/working/out/ 2>/dev/null; cd /kaggle/working && zip -qr out.zip out && ls -lh out.zip
!tail -3 results/cost_ledger.jsonl 2>/dev/null || echo "no paid calls" 